# Pelican file events: live EarthScope GNSS data

This notebook subscribes to a Pelican namespace, and plots each new
object as it appears. The data is EarthScope GNSS displacement:
east, north and up, one file per minute.

Nothing is downloaded. Each object is read straight from the
federation into memory, and the NDP Endpoint is not in the data path.

Everything here goes through `ndp-ep`. There is no STOMP, no
WebSocket handling and no `asyncio` to write.

In [2]:
%pip install "ndp-ep[pelican]"

Note: you may need to restart the kernel to use updated packages.


In [3]:
import logging

logging.basicConfig(level=logging.INFO)

## Configuration

`CLIENT_ID` identifies your subscriber, and the event server keeps one
queue per id. Sharing an id with another reader is worse than it
sounds: you compete for the same events, and you inherit whatever that
queue has already accumulated. The default below derives an id from
your local username so that does not happen. Keep it stable between
runs — a new id every time leaves an abandoned queue behind on the
server.

`EVENT_SOURCE` is the namespace to watch. It is a rolling window: it
holds the last 100 objects, roughly the last hour and forty minutes,
and the publisher adds one per minute while the oldest is deleted. An
event can therefore point at an object that no longer exists, which is
why the loop further down tolerates a failed read.

The credentials are checked by the event server against its own store;
they are unrelated to the Endpoint token, and will be replaced by an
access token later.

In [4]:
import getpass

# One queue per id, so this must not be shared with other readers.
CLIENT_ID = f"{getpass.getuser()}-pelican-demo"
EVENT_SOURCE = "osdf/vdc/public/pelican_protocol"

ENDPOINT_URL = "http://155.101.6.191:8003"
EVENT_USERNAME = "your-username"
EVENT_PASSWORD = "your-password"

## Connect and subscribe

One client object covers both halves: the subscription that tells you
an object appeared, and the reads that fetch it.

In [5]:
from ndp_ep import APIClient

client = APIClient(base_url=ENDPOINT_URL)

subscription = client.subscribe_pelican(
    EVENT_SOURCE,
    client_id=CLIENT_ID,
    username=EVENT_USERNAME,
    password=EVENT_PASSWORD,
)

subscription.wait_until_connected(timeout=30)
subscription.status

C:\Users\rbard\code\ndp-ep-py\ndp_ep\pelican_events_method.py:569: FutureWarning: Pelican event subscriptions authenticate with a username and password held by the event server. This is a temporary arrangement and will be replaced by an access token; expect these arguments to change.
  subscriber = PelicanSubscription(
INFO:ndp_ep.pelican_events_method:Subscribed to rbard-pelican-demo/osdf/vdc/public/pelican_protocol as rbard-pelican-demo


{'state': 'connected',
 'last_error': '',
 'session': {'server': 'stomp-playground',
  'session': '6f62fa0c-36cc-4068-ad6c-99446e8414e3',
  'version': '1.2'},
 'subscription': {'id': 'rbard-pelican-demo',
  'destination': 'rbard-pelican-demo/osdf/vdc/public/pelican_protocol',
  'active': True},
 'config': {'url': 'wss://stomp-server.chtcdev.chtc.io/ws',
  'event_source': 'osdf/vdc/public/pelican_protocol',
  'client_id': 'rbard-pelican-demo',
  'heartbeat': 10000,
  'reconnect': True,
  'authenticated': True},
 'metrics': {'connection_attempts': 1,
  'sessions_established': 1,
  'connection_failures': 0,
  'messages_received': 0,
  'events_delivered': 0,
  'duplicates_suppressed': 0,
  'unparseable_messages': 0,
  'server_errors': 0,
  'acks_sent': 0},
 'pending': 0,
 'processed_total': 6}

## Plot each file as it arrives

`subscription.events(timeout=...)` blocks until the next event and
yields it once. Redeliveries are suppressed against a record on disk,
so no bookkeeping is needed here and restarting the notebook will not
reprocess what it already plotted.

The publisher writes roughly one file per minute, but a newly
connected subscriber usually sees nothing for the first few minutes:
measured runs waited between two and four. The server then delivers
what it has queued in a burst, so those first events arrive seconds
apart rather than a minute apart. Expect the cell below to take
several minutes for three events, most of it before the first one,
and do not read the initial silence as a failure.

`EVENT_TIMEOUT` bounds the wait for each individual event, so the cell
always ends instead of hanging if the publisher goes quiet. It has to
be comfortably larger than that startup delay. Raise `MAX_EVENTS` to
keep the cell running longer.

A new subscriber does not replay the namespace's history, but the
first burst can include objects written shortly before it connected.

In [6]:
%pip install plotly anywidget pandas

Note: you may need to restart the kernel to use updated packages.


In [7]:
import io

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

MAX_POINTS = 600
MAX_EVENTS = 3
EVENT_TIMEOUT = 420  # seconds; well above the startup delay

all_data = pd.DataFrame()


def create_figure(title, color):
    fig = go.FigureWidget()
    fig.add_scatter(mode="lines", line=dict(color=color, width=2),
                    name=title)
    fig.update_layout(
        title=title,
        template="plotly_white",
        height=250,
        margin=dict(l=50, r=30, t=40, b=40),
        xaxis_title="Time",
        yaxis_title=title,
    )
    return fig


east_fig = create_figure("East", "royalblue")
north_fig = create_figure("North", "green")
up_fig = create_figure("Up", "firebrick")

display(east_fig)
display(north_fig)
display(up_fig)

received = 0

for event in subscription.events(timeout=EVENT_TIMEOUT):

    try:
        raw = client.pelican_read(event.url)
    except ValueError as exc:
        # The namespace keeps only the last 100 objects. An event can
        # arrive for one that has already been rotated out.
        print(f"Skipped {event.name}: {exc}")
        continue

    print(f"Received {event.name}")

    df = pd.read_csv(io.BytesIO(raw))
    df["datetime"] = pd.to_datetime(df["time"], unit="ms", utc=True)

    all_data = pd.concat([all_data, df], ignore_index=True)
    if len(all_data) > MAX_POINTS:
        all_data = all_data.iloc[-MAX_POINTS:].copy()

    x = all_data["datetime"]
    for figure, column in (
        (east_fig, "east"),
        (north_fig, "north"),
        (up_fig, "up"),
    ):
        with figure.batch_update():
            figure.data[0].x = x
            figure.data[0].y = all_data[column]

    received += 1
    if received >= MAX_EVENTS:
        break

FigureWidget({
    'data': [{'line': {'color': 'royalblue', 'width': 2},
              'mode': 'lines',
              'name': 'East',
              'type': 'scatter',
              'uid': 'd136b488-c03a-430d-b021-bb82d58ffc07'}],
    'layout': {'height': 250,
               'margin': {'b': 40, 'l': 50, 'r': 30, 't': 40},
               'template': '...',
               'title': {'text': 'East'},
               'xaxis': {'title': {'text': 'Time'}},
               'yaxis': {'title': {'text': 'East'}}}
})

FigureWidget({
    'data': [{'line': {'color': 'green', 'width': 2},
              'mode': 'lines',
              'name': 'North',
              'type': 'scatter',
              'uid': 'db34639a-69ce-4010-918d-803bb3c8f738'}],
    'layout': {'height': 250,
               'margin': {'b': 40, 'l': 50, 'r': 30, 't': 40},
               'template': '...',
               'title': {'text': 'North'},
               'xaxis': {'title': {'text': 'Time'}},
               'yaxis': {'title': {'text': 'North'}}}
})

FigureWidget({
    'data': [{'line': {'color': 'firebrick', 'width': 2},
              'mode': 'lines',
              'name': 'Up',
              'type': 'scatter',
              'uid': 'de8b16fc-ff9a-4fe3-b6c2-69764dc2b99f'}],
    'layout': {'height': 250,
               'margin': {'b': 40, 'l': 50, 'r': 30, 't': 40},
               'template': '...',
               'title': {'text': 'Up'},
               'xaxis': {'title': {'text': 'Time'}},
               'yaxis': {'title': {'text': 'Up'}}}
})

Received AGMT.CI.LY_.20_c280.csv
Received AGMT.CI.LY_.20_c281.csv
Received AGMT.CI.LY_.20_c282.csv


## All three components on one figure

In [8]:
combined = go.FigureWidget()

for column, color in (("east", "royalblue"),
                     ("north", "green"),
                     ("up", "firebrick")):
    combined.add_scatter(
        x=all_data["datetime"],
        y=all_data[column],
        name=column.capitalize(),
        line=dict(color=color, width=2),
    )

combined.update_layout(template="plotly_white", height=350,
                       xaxis_title="Time")
display(combined)

FigureWidget({
    'data': [{'line': {'color': 'royalblue', 'width': 2},
              'name': 'East',
              'type': 'scatter',
              'uid': '8ce41f73-9f86-4584-be90-ae1e85bcdafd',
              'x': array(['2024-12-03T04:43:31.000', '2024-12-03T04:43:32.000',
                          '2024-12-03T04:43:33.000', '2024-12-03T04:43:34.000',
                          '2024-12-03T04:43:35.000', '2024-12-03T04:43:36.000',
                          '2024-12-03T04:43:37.000', '2024-12-03T04:43:38.000',
                          '2024-12-03T04:43:39.000', '2024-12-03T04:43:40.000',
                          '2024-12-03T04:43:41.000', '2024-12-03T04:43:42.000',
                          '2024-12-03T04:43:43.000', '2024-12-03T04:43:44.000',
                          '2024-12-03T04:43:45.000', '2024-12-03T04:43:46.000',
                          '2024-12-03T04:43:47.000', '2024-12-03T04:43:48.000',
                          '2024-12-03T04:43:49.000', '2024-12-03T04:43:50.000',
   

## Browsing without subscribing

The namespace can be listed and read directly, with no subscription
involved. References are accepted in any of the spellings that turn
up in practice: a bare path, `osdf://...`, or `pelican://host/...`.

`pelican_list` returns names in lexicographic order, not chronological
order: `AGMT.CI.LY_.20_c100.csv` sorts before `AGMT.CI.LY_.20_c99.csv`.
The end of the list is therefore not the newest data. Pick a name
explicitly, as below, and use an event's `url` when what you want is
the most recent object.

In [9]:
objects = client.pelican_list(EVENT_SOURCE)

print(f"{len(objects)} objects")

sample = objects[0]
sample

100 objects


'/vdc/public/pelican_protocol/AGMT.CI.LY_.20_c190.csv'

In [10]:
raw = client.pelican_read(sample)

print(raw[:200].decode())

time,east,north,up,sigEE,sigNN,sigUU,qChannel
1733195537000,-0.092,0.033,0.107,0.206,0.156,0.234,2707031
1733195538000,-0.088,0.028,0.102,0.205,0.155,0.234,2707031
1733195539000,-0.087,0.03,0.096,0


## Downloading to disk

`pelican_read` keeps the object in memory. Use `pelican_fetch` when
a file on disk is what you want; an existing target is an error
rather than a silent overwrite.

In [11]:
path = client.pelican_fetch(sample, "./data/")

print(path, path.stat().st_size, "bytes")

data\AGMT.CI.LY_.20_c190.csv 3598 bytes


## The raw events

Each event carries the object's name, a reference ready to pass to
`pelican_read`, its size, and the modification time reported by the
server.

Passing a `timeout` bounds the wait, so the cell ends even if the
publisher goes quiet. Keep it well above the few-minute startup delay:
on a subscription that has just connected, a shorter bound returns
nothing and looks like a failure rather than a timeout.

In [12]:
for received, event in enumerate(
    subscription.events(timeout=EVENT_TIMEOUT), 1
):
    print(event.name, event.size, event.mod_time)
    print("   ", event.url)
    if received >= 2:
        break

AGMT.CI.LY_.20_c283.csv 3628 2026-08-27T18:17:02Z
    osdf://vdc/public/pelican_protocol/AGMT.CI.LY_.20_c283.csv
AGMT.CI.LY_.20_c284.csv 3630 2026-08-27T18:18:02Z
    osdf://vdc/public/pelican_protocol/AGMT.CI.LY_.20_c284.csv


## Closing

Closing performs the WebSocket closing handshake, so the event server
does not sit on a half-open connection. Using the subscription as a
context manager (`with client.subscribe_pelican(...) as subscription:`)
does this automatically.

In [13]:
subscription.close()
subscription.status["metrics"]

{'connection_attempts': 1,
 'sessions_established': 1,
 'connection_failures': 0,
 'messages_received': 5,
 'events_delivered': 5,
 'duplicates_suppressed': 0,
 'unparseable_messages': 0,
 'server_errors': 0,
 'acks_sent': 5}